# N16. Bright border training, and the control arm trained without occlusion

Two experiments both reviewers asked for. Neither has been run before.

**Question 1. Is the border effect something training can remove?**
Covering only the border, with all 36 data cells visible, drops the published decoder from
92 percent to 49 percent as the paint goes from black to white (N8). N11 showed that fixing
the brightness scale at test time removes about 45 percent of that spread. That is an
intervention at test time. This notebook trains decoders on the exact thing:

* `N`  our recipe with no paint at all
* `BB` border ring painted at a random brightness, no data cell ever hidden
* `BD` data region only, painted at a random brightness, as a control

The decisive comparison is **BB against N**: same recipe, same markers, same order, and the
only difference is that BB saw painted borders. N8 already showed that arms trained on edge
strips (B and E) are much less border sensitive than the published decoder (spread 16.6 and
10.8 points against 43.7), because their strips also cover the border. So comparing BB with E
would test something that is already mostly solved. N is the clean baseline. Comparing N with
the published decoder also shows whether our recipe, trained clean, is as border sensitive as
the authors' decoder.

**Question 2. Is finding 4 caused by occlusion training, or by our training recipe?**
The published decoder beats our arm B on black occluders. But the two differ in two ways:
B saw grey occluders, and B was trained by our code rather than the authors'. This notebook
trains `N`: our exact code, data and schedule, with no occlusion at all.

* published decoder against N isolates the recipe, since neither saw occlusion
* N against B isolates occlusion training, since the recipe is now identical

**Why the comparisons are unusually clean.** Every new arm has one output, like B and E.
With the deterministic code from N14, arms in the same group start from the same weights,
see the same markers in the same order with the same augmentation, and differ only in what
was painted on the training crops. So seed s of BB is compared with seed s of N, which
removes most of the seed noise that made N15's comparisons inconclusive. The new training
sets are checked to have the same 64,840 crops, in the same order, with identical labels.

**Rules for reading the results, fixed before running.**
A paired difference over the 5 seeds is `differs` when its 95 percent interval excludes zero,
`equivalent` when the whole interval lies within plus or minus 1 point, and `inconclusive`
otherwise. Inconclusive is never written as "no effect".

* Q1: if BB minus N on the border spread `differs` with BB lower, and BB's spread is below 5
  points, border training removes the border effect. If it differs but stays at 5 or above, it
  reduces it. If BB minus N does not differ, border training does not detectably change it.
  The verdict also says whether N itself is border sensitive, since without that the
  comparison has nothing to remove.
* Q2: if N minus B at black `differs` with N higher, occlusion training at grey makes black
  worse and finding 4 holds with the recipe controlled. If N minus B is not different, the
  published decoder's lead over B on black comes from the recipe, and finding 4 must change.

**Evaluation protocol, frozen.** Development cameras 1 to 5, every 6th frame, 12,276 markers,
the same set as N8, N11, N14 and N15. Soft decoding for every one output arm. Fair tie breaking
whenever hard decoding is reported. Camera 6 is not touched.

**Before running: Runtime, Restart session, then Run all.** Needs the arm B and arm E
checkpoints from N14 in `ckpt_n14`. Trains 15 new runs, about 40 minutes on an L4.


In [1]:
!pip -q install opencv-contrib-python==4.10.0.84 h5py 2>&1 | tail -1

In [2]:
%%writefile deeparuco_port.py
"""
Port the pretrained DeepArUco++ decoder from Keras .h5 to PyTorch, without TensorFlow.

The published decoder (models/dec_new.h5 in AVAuco/deeparuco) is a Keras 2.11 model:

    Input(32, 32, 1)
    Conv2D(8, 5x5, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(4, 3x3, valid) -> PReLU -> MaxPool2D(2)
    Conv2D(1, 1x1, valid, sigmoid)   ->  (6, 6, 1)

Only h5py is needed to read the weights, so the repo's Python 3.9 + TensorFlow
environment is never required for baseline inference.

IMPORTANT PORTING DETAIL
    Keras PReLU defaults to shared_axes=None, so alpha is PER-ELEMENT, not per-channel:
    the stored alphas have shape (28, 28, 8) and (12, 12, 4). torch.nn.PReLU only
    supports per-channel alpha, so using it here silently produces a different model.
    SpatialPReLU below keeps the full alpha tensor.

Layout conversions:
    Keras conv kernel (kh, kw, in, out) -> torch (out, in, kh, kw)
    Keras PReLU alpha (h, w, c)         -> torch (c, h, w)
    Keras 'valid' padding               -> torch padding=0

Usage:
    from deeparuco_port import load_weights, NumpyDecoder, TorchDecoder, verify_port
    verify_port('models/dec_new.h5')          # run this first, once
    dec = TorchDecoder.from_h5('models/dec_new.h5')
    probs = dec(x)                            # x: (N, 1, 32, 32) float in [0, 1]

Upstream code is AGPL-3.0. This module reads a weights file; it does not copy source.
Cite Berral-Soler et al., Image and Vision Computing 152 (2024) 105313.
"""

import numpy as np

DEFAULT_H5 = "models/dec_new.h5"


# --------------------------------------------------------------------------- weights

def load_weights(h5_path=DEFAULT_H5):
    """Read decoder weights from the Keras .h5. Returns a dict of numpy arrays."""
    import h5py

    store = {}
    with h5py.File(h5_path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset) and name.startswith("model_weights/"):
                store[name] = np.array(obj)
        f.visititems(visit)

    def pick(layer, suffix):
        hits = [k for k in store if f"/{layer}/" in k and k.endswith(suffix)]
        if len(hits) != 1:
            raise KeyError(f"expected one match for {layer}:{suffix}, got {hits}")
        return store[hits[0]]

    w = {
        "c1_w": pick("conv2d", "kernel:0"),   "c1_b": pick("conv2d", "bias:0"),
        "p1_a": pick("p_re_lu", "alpha:0"),
        "c2_w": pick("conv2d_1", "kernel:0"), "c2_b": pick("conv2d_1", "bias:0"),
        "p2_a": pick("p_re_lu_1", "alpha:0"),
        "c3_w": pick("conv2d_2", "kernel:0"), "c3_b": pick("conv2d_2", "bias:0"),
    }
    expect = {"c1_w": (5,5,1,8), "c1_b": (8,), "p1_a": (28,28,8),
              "c2_w": (3,3,8,4), "c2_b": (4,), "p2_a": (12,12,4),
              "c3_w": (1,1,4,1), "c3_b": (1,)}
    for k, shape in expect.items():
        assert w[k].shape == shape, f"{k}: got {w[k].shape}, expected {shape}"
    return w


# ------------------------------------------------------------------- numpy reference

class NumpyDecoder:
    """Reference implementation. Verified against synthetic markers: 4/5 exact."""

    def __init__(self, weights):
        self.w = weights

    @classmethod
    def from_h5(cls, path=DEFAULT_H5):
        return cls(load_weights(path))

    @staticmethod
    def _conv(x, w, b):
        kh, kw, _, co = w.shape
        H, W, _ = x.shape
        out = np.zeros((H - kh + 1, W - kw + 1, co), np.float32)
        for i in range(out.shape[0]):
            for j in range(out.shape[1]):
                out[i, j] = np.tensordot(
                    x[i:i+kh, j:j+kw, :], w, axes=([0, 1, 2], [0, 1, 2])) + b
        return out

    @staticmethod
    def _pool(x):
        h, w, c = x.shape
        return x.reshape(h // 2, 2, w // 2, 2, c).max(axis=(1, 3))

    def __call__(self, marker32):
        """marker32: (32, 32) float in [0, 1]. Returns (6, 6) bit probabilities."""
        w = self.w
        x = np.asarray(marker32, np.float32)[..., None]
        x = self._conv(x, w["c1_w"], w["c1_b"])
        x = np.where(x > 0, x, w["p1_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c2_w"], w["c2_b"])
        x = np.where(x > 0, x, w["p2_a"] * x)
        x = self._pool(x)
        x = self._conv(x, w["c3_w"], w["c3_b"])
        return (1.0 / (1.0 + np.exp(-np.clip(x[..., 0], -60, 60)))).astype(np.float32)


# -------------------------------------------------------------------------- pytorch

def _torch():
    import torch
    import torch.nn as nn
    return torch, nn


class SpatialPReLU:
    """Created via _build so torch is imported lazily. See module docstring."""


def TorchDecoder_factory():
    torch, nn = _torch()

    class _SpatialPReLU(nn.Module):
        """Per-element PReLU, matching Keras PReLU(shared_axes=None)."""
        def __init__(self, alpha_chw):
            super().__init__()
            self.alpha = nn.Parameter(torch.tensor(alpha_chw, dtype=torch.float32))

        def forward(self, x):
            return torch.where(x > 0, x, self.alpha.unsqueeze(0) * x)

    class _TorchDecoder(nn.Module):
        def __init__(self, weights=None):
            super().__init__()
            self.c1 = nn.Conv2d(1, 8, 5)
            self.c2 = nn.Conv2d(8, 4, 3)
            self.c3 = nn.Conv2d(4, 1, 1)
            self.pool = nn.MaxPool2d(2)
            a1 = np.zeros((8, 28, 28), np.float32) if weights is None \
                 else np.transpose(weights["p1_a"], (2, 0, 1))
            a2 = np.zeros((4, 12, 12), np.float32) if weights is None \
                 else np.transpose(weights["p2_a"], (2, 0, 1))
            self.p1, self.p2 = _SpatialPReLU(a1), _SpatialPReLU(a2)
            if weights is not None:
                self.load_keras(weights)

        def load_keras(self, w):
            with torch.no_grad():
                for conv, kw, kb in ((self.c1, "c1_w", "c1_b"),
                                     (self.c2, "c2_w", "c2_b"),
                                     (self.c3, "c3_w", "c3_b")):
                    conv.weight.copy_(torch.tensor(np.transpose(w[kw], (3, 2, 0, 1))))
                    conv.bias.copy_(torch.tensor(w[kb]))

        @classmethod
        def from_h5(cls, path=DEFAULT_H5):
            return cls(load_weights(path))

        def forward(self, x):
            """x: (N, 1, 32, 32) in [0, 1]. Returns (N, 6, 6) bit probabilities."""
            x = self.pool(self.p1(self.c1(x)))
            x = self.pool(self.p2(self.c2(x)))
            return torch.sigmoid(self.c3(x)).squeeze(1)

    return _TorchDecoder


def __getattr__(name):
    if name == "TorchDecoder":
        return TorchDecoder_factory()
    raise AttributeError(name)


# --------------------------------------------------------------------------- verify

def verify_port(h5_path=DEFAULT_H5, n=16, tol=1e-4, verbose=True):
    """Assert the PyTorch port matches the numpy reference. Run once before use.

    The numpy path is validated; the torch path is validated against it here rather
    than assumed. Do not skip this.
    """
    torch, _ = _torch()
    w = load_weights(h5_path)
    ref = NumpyDecoder(w)
    net = TorchDecoder_factory()(w).eval()

    rng = np.random.default_rng(0)
    x = rng.random((n, 32, 32)).astype(np.float32)
    with torch.no_grad():
        got = net(torch.tensor(x).unsqueeze(1)).numpy()
    want = np.stack([ref(xi) for xi in x])

    err = float(np.abs(got - want).max())
    if verbose:
        print(f"port check: {n} random inputs, max abs difference {err:.2e} (tol {tol:g})")
    assert err < tol, f"PyTorch port disagrees with numpy reference (max diff {err:.2e})"
    if verbose:
        print("PyTorch port matches the numpy reference.")
    return err


def decode_ids(probs, id_to_bits, n_markers=250):
    """Hard-decision decode: nearest codeword by Hamming distance over 4 rotations.

    probs: (6, 6) bit probabilities. Returns (marker_id, rotation, hamming).
    Replace with soft log-likelihood decoding to use the confidences.
    """
    obs = (np.asarray(probs) > 0.5).astype(np.uint8)
    best = (None, None, 99)
    for mid in range(n_markers):
        ref = np.asarray(id_to_bits(mid), np.uint8).reshape(6, 6)
        for rot in range(4):
            h = int(np.sum(np.rot90(obs, rot) != ref))
            if h < best[2]:
                best = (mid, rot, h)
    return best


if __name__ == "__main__":
    import sys
    path = sys.argv[1] if len(sys.argv) > 1 else DEFAULT_H5
    w = load_weights(path)
    print("weights loaded:", {k: v.shape for k, v in w.items()})
    try:
        verify_port(path)
    except ImportError:
        print("torch not installed; numpy reference verified only")


Writing deeparuco_port.py


In [3]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # deterministic cuBLAS, must be set before torch
import sys, json, glob, zlib, subprocess, time, itertools
import numpy as np, cv2, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, WORK = '/content/drive/MyDrive/aruco_p2', '/content/work'
except ImportError:
    BASE = os.environ.get('ARUCO_P2_BASE',
                          os.path.join(os.path.expanduser('~'), 'aruco_p2')).replace('\\', '/')
    WORK = f'{BASE}/work'
ART, CACHE = f'{BASE}/artifacts', f'{BASE}/archives'
os.makedirs(WORK, exist_ok=True); sys.path.insert(0, os.getcwd())
assert os.path.exists(f'{ART}/splits.json'), f'{ART}/splits.json missing, run N0 first'

REPO = f'{WORK}/deeparuco'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/AVAuco/deeparuco.git',REPO],check=True)
sys.path.insert(0, REPO)
from impl.aruco import id_to_bits
from deeparuco_port import load_weights, NumpyDecoder

DEV_ = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
splits = json.load(open(f'{ART}/splits.json'))
DEVV, HELDOUT, STRIDE = splits['dev'], splits['heldout'][0], splits['benchmark_stride']
Wk  = load_weights(f'{REPO}/models/dec_new.h5')
CB  = np.stack([np.asarray(id_to_bits(i),np.uint8).reshape(6,6) for i in range(250)])
ROT = np.stack([np.rot90(CB,k,axes=(1,2)).reshape(250,36) for k in range(4)]).astype(np.float32)

SHADOW = f'{WORK}/shadow'; os.makedirs(SHADOW, exist_ok=True)
for i in range(1,7):
    if not os.path.isdir(f'{SHADOW}/video_{i}'):
        tar = f'{CACHE}/video_{i}.tar.gz'
        assert os.path.exists(tar), f'{tar} missing, see the download cell in N0'
        subprocess.run(['tar','-xzf',tar,'-C',SHADOW], check=True)

N0M = json.load(open(f'{ART}/manifest.json'))
for vid in DEVV + [HELDOUT]:
    n = len(glob.glob(f'{SHADOW}/{vid}/*.png'))
    assert n == N0M['shadow_aruco']['frames'][vid], f'{vid}: {n} frames on disk, manifest says other'
print('torch', torch.__version__, '| device', DEV_)
if DEV_.type != 'cuda':
    print('\n*** THIS IS A CPU RUNTIME. One training run takes about 40 minutes here')
    print('*** and about 2 minutes on a GPU. Use Runtime, Change runtime type, T4 or L4,')
    print('*** then rerun every cell from the top before training.')
print('dev:', DEVV, '| held out:', HELDOUT, '| stride', STRIDE)


Mounted at /content/drive
torch 2.14.1+cu130 | device cuda
dev: ['video_1', 'video_2', 'video_3', 'video_4', 'video_5'] | held out: video_6 | stride 6


In [4]:
r_int = lambda v: int(round(v))
VALUES = {'grey':127, 'black':0, 'white':255}

def ordered_corners(xs, ys):
    cx, cy = np.mean(xs), np.mean(ys)
    idx = np.argsort(np.arctan2(xs-cx, ys-cy))
    return [v for i in idx for v in (xs[i], ys[i])]

def warp32(crop, c, t=32):
    dst = np.array([[0,0],[0,t-1],[t-1,t-1],[t-1,0]], np.float32)
    src = np.clip(np.array([[c[0]*63,c[1]*63],[c[2]*63,c[3]*63],
                            [c[4]*63,c[5]*63],[c[6]*63,c[7]*63]]),0,63).astype(np.float32)
    return cv2.warpPerspective(crop, cv2.getPerspectiveTransform(src,dst), (t,t))

def get_corners(m):
    c = m['corners']
    if isinstance(c[0], dict): return np.array([[p['x'],p['y']] for p in c], np.float32)
    return np.asarray(c, np.float32).reshape(4,2)

def cell_occlusion(m32):
    s = 32/8.0; out = np.zeros((6,6), bool)
    for r in range(6):
        for c in range(6):
            out[r,c] = m32[int((r+1)*s):int((r+2)*s), int((c+1)*s):int((c+2)*s)].mean() > 127.5
    return out

def prepare(img, mask, corners, scale=None):
    x, y = corners[:,0], corners[:,1]
    mnx, mxx, mny, mxy = x.min(), x.max(), y.min(), y.max()
    ex, ey = 0.2*(mxx-mnx)+0.5, 0.2*(mxy-mny)+0.5
    mnx, mny = max(mnx-ex,0), max(mny-ey,0)
    mxx = min(mxx+ex, img.shape[1]-1); mxy = min(mxy+ey, img.shape[0]-1)
    if mxx-mnx < 8 or mxy-mny < 8: return None, None
    sl = (slice(r_int(mny), r_int(mxy)), slice(r_int(mnx), r_int(mxx)))
    crop = cv2.resize(img[sl], (64,64)).astype(np.float32)
    lo, hi = (crop.min(), crop.max()) if scale is None else scale
    crop = np.clip((crop - lo)/(hi - lo + 1e-9), 0, 1)
    nc = ordered_corners((x-mnx)/(mxx-mnx), (y-mny)/(mxy-mny))
    m32 = warp32(cv2.resize(mask[sl],(64,64)), nc) if mask.any() else np.zeros((32,32), np.uint8)
    return warp32(crop, nc), cell_occlusion(m32)

def occluder_box(img, corners, level, position, key, mid):
    if level <= 0: return None
    x, y = corners[:,0], corners[:,1]
    x0,x1,y0,y1 = x.min(),x.max(),y.min(),y.max(); w,h = x1-x0, y1-y0
    if position == 'edge':
        side = int(np.random.default_rng(zlib.crc32(f'occlusion|{level}|{key}|{mid}'.encode())).integers(0,4))
        box = [(x0,y0,x0+level*w,y1),(x1-level*w,y0,x1,y1),
               (x0,y0,x1,y0+level*h),(x0,y1-level*h,x1,y1)][side]
    else:
        cx, cy = (x0+x1)/2, (y0+y1)/2
        box = (cx-level*w/2, cy-level*h/2, cx+level*w/2, cy+level*h/2)
    a,b,c,d = [int(round(v)) for v in box]
    a,b = max(a,0),max(b,0); c,d = min(c,img.shape[1]),min(d,img.shape[0])
    return (a,b,c,d) if (c>a and d>b) else None

def apply_occluder(img, box, value):
    if box is None: return img, np.zeros(img.shape[:2], np.uint8)
    if isinstance(value, str): value = VALUES[value]
    a,b,c,d = box
    out = img.copy(); mk = np.zeros(img.shape[:2], np.uint8)
    out[b:d,a:c] = int(value); mk[b:d,a:c] = 255
    return out, mk

def pairs_for(vid):
    out = []
    for f in sorted(glob.glob(f'{SHADOW}/{vid}/*.png')):
        j = f'{SHADOW}/{vid}/corrected_annotations/{os.path.splitext(os.path.basename(f))[0]}.json'
        if os.path.exists(j): out.append((f,j))
    return out

class SpatialPReLU(nn.Module):
    def __init__(s,c,h,w): super().__init__(); s.a = nn.Parameter(torch.zeros(c,h,w))
    def forward(s,x): return torch.where(x>0, x, s.a.unsqueeze(0)*x)

class Decoder(nn.Module):
    """8 and 4 filters, as in the released code. pool is 'max' or 'avg'."""
    def __init__(s, out_ch=1, pool='max'):
        super().__init__()
        s.c1=nn.Conv2d(1,8,5); s.p1=SpatialPReLU(8,28,28)
        s.c2=nn.Conv2d(8,4,3); s.p2=SpatialPReLU(4,12,12)
        s.c3=nn.Conv2d(4,out_ch,1); s.out_ch=out_ch
        s.pool = F.max_pool2d if pool=='max' else F.avg_pool2d
    def forward(s,x):
        x=s.pool(s.p1(s.c1(x)),2); x=s.pool(s.p2(s.c2(x)),2); x=s.c3(x)
        return (x[:,0], x[:,1]) if s.out_ch==2 else (x[:,0], None)
print('pipeline ready')


pipeline ready


In [5]:
def _conv(x,w,b):
    kh,kw,_,_ = w.shape
    return np.einsum('nohckl,klci->nohi', sliding_window_view(x,(kh,kw),axis=(1,2)), w, optimize=True)+b
def _pool(x):
    N,H,Wd,Cc = x.shape; return x.reshape(N,H//2,2,Wd//2,2,Cc).max(axis=(2,4))

def decode_published(x8, chunk=4096):
    out=[]
    for s in range(0,len(x8),chunk):
        x = x8[s:s+chunk].astype(np.float32)[...,None]/255.0
        x = _pool(np.where((y:=_conv(x,Wk['c1_w'],Wk['c1_b']))>0,y,Wk['p1_a']*y))
        x = _pool(np.where((y:=_conv(x,Wk['c2_w'],Wk['c2_b']))>0,y,Wk['p2_a']*y))
        out.append(1/(1+np.exp(-np.clip(_conv(x,Wk['c3_w'],Wk['c3_b'])[...,0],-60,60))))
    return np.concatenate(out).astype(np.float32)

def decode_torch(net, x8, chunk=8192):
    ps, es = [], []
    with torch.no_grad():
        for s in range(0,len(x8),chunk):
            xb = torch.tensor(x8[s:s+chunk], dtype=torch.float32).unsqueeze(1).to(DEV_)/255.0
            lo, el = net(xb)
            ps.append(torch.sigmoid(lo).cpu().numpy())
            es.append(torch.sigmoid(el).cpu().numpy() if el is not None else np.zeros((len(lo),6,6),np.float32))
    return np.concatenate(ps), np.concatenate(es)

def dec_soft(P, Wt=None):
    LP = np.log(np.clip(P,1e-6,1)).reshape(len(P),36)
    LQ = np.log(np.clip(1-P,1e-6,1)).reshape(len(P),36)
    if Wt is not None:
        w = Wt.reshape(len(P),36); LP, LQ = LP*w, LQ*w
    return np.stack([LP@ROT[k].T + LQ@(1-ROT[k]).T for k in range(4)]).max(0).argmax(1)
def hamming_w(P, W=None):
    B = (P.reshape(len(P),36) > 0.5).astype(np.float32)
    W = np.ones_like(B) if W is None else W.reshape(len(P),36).astype(np.float32)
    return np.stack([(B*W)@(1-ROT[k]).T + ((1-B)*W)@ROT[k].T for k in range(4)]).min(0)

def hard_expected(P, ids, W=None):
    """Exact expected accuracy with fair random tie breaking, per marker (0 to 1)."""
    D = hamming_w(P, W); tied = D == D.min(1, keepdims=True)
    return tied[np.arange(len(ids)), ids] / tied.sum(1)
print('decoders ready')


decoders ready


In [6]:
def paired(a_ok, b_ok, groups, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    uf, inv = np.unique(groups, return_inverse=True)
    by = [np.where(inv==i)[0] for i in range(len(uf))]
    obs = a_ok.mean() - b_ok.mean()
    d = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([by[p] for p in rng.integers(0, len(by), len(by))])
        d[i] = a_ok[idx].mean() - b_ok[idx].mean()
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = 2*min((d <= 0).mean(), (d >= 0).mean())
    return float(obs), float(lo), float(hi), max(float(p), 1/n_boot)
from scipy import stats as sst

def welch(a, b):
    """Unpaired difference of means across seeds with a Welch t interval."""
    a, b = np.asarray(a, float), np.asarray(b, float); m = a.mean() - b.mean()
    va, vb = a.var(ddof=1)/len(a), b.var(ddof=1)/len(b); se = np.sqrt(va + vb)
    if se == 0: return float(m), float(m), float(m)
    df = (va + vb)**2 / (va**2/(len(a)-1) + vb**2/(len(b)-1))
    h = sst.t.ppf(0.975, df) * se
    return float(m), float(m - h), float(m + h)
print('statistics ready')


statistics ready


In [7]:
# ------------------------------------------------------------------ border conditions
# The border ring is the outer cell of the 8 by 8 marker; the data region is the inner 6 by 6.
# These are the same definitions and the same five brightnesses as N8 and N11.
INTENS = [0, 64, 127, 192, 255]
BORDER_LEVEL = 0.50            # strip size for data_only and both, as in N8

def region_masks(corners, shape):
    c = corners.mean(axis=0)
    inner = c + (corners - c) * (6.0/8.0)
    m_out = np.zeros(shape, np.uint8); cv2.fillPoly(m_out, [corners.astype(np.int32)], 255)
    m_in  = np.zeros(shape, np.uint8); cv2.fillPoly(m_in,  [inner.astype(np.int32)], 255)
    return cv2.bitwise_and(m_out, cv2.bitwise_not(m_in)), m_in

def make_condition(img, corners, cond, value, key, mid, level=BORDER_LEVEL):
    if cond == 'clean':
        return img, np.zeros(img.shape[:2], np.uint8)
    border, data = region_masks(corners, img.shape[:2])
    if cond == 'border_only':
        paint = border
    else:
        box = occluder_box(img, corners, level, 'edge', key, mid)
        if box is None: return None, None
        a,b,c,d = box
        strip = np.zeros(img.shape[:2], np.uint8); strip[b:d, a:c] = 255
        paint = cv2.bitwise_and(strip, data) if cond == 'data_only' else strip
    out = img.copy(); out[paint > 0] = int(value)
    return out, paint

def crop_scale(img, corners):
    """The (min, max) a clean crop of this marker has. Used by the N11 oracle."""
    x, y = corners[:,0], corners[:,1]
    mnx, mxx, mny, mxy = x.min(), x.max(), y.min(), y.max()
    ex, ey = 0.2*(mxx-mnx)+0.5, 0.2*(mxy-mny)+0.5
    mnx, mny = max(mnx-ex,0), max(mny-ey,0)
    mxx = min(mxx+ex, img.shape[1]-1); mxy = min(mxy+ey, img.shape[0]-1)
    if mxx-mnx < 8 or mxy-mny < 8: return None
    c = cv2.resize(img[r_int(mny):r_int(mxy), r_int(mnx):r_int(mxx)], (64,64)).astype(np.float32)
    return float(c.min()), float(c.max())

def paired_t(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); m = d.mean()
    if len(d) < 2 or d.std(ddof=1) == 0: return float(m), float(m), float(m)
    h = sst.t.ppf(0.975, len(d)-1) * d.std(ddof=1) / np.sqrt(len(d))
    return float(m), float(m-h), float(m+h)

def one_sample_t(v):
    v = np.asarray(v, float); m = v.mean()
    if len(v) < 2 or v.std(ddof=1) == 0: return float(m), float(m), float(m)
    h = sst.t.ppf(0.975, len(v)-1) * v.std(ddof=1) / np.sqrt(len(v))
    return float(m), float(m-h), float(m+h)

def verdict(lo, hi):
    if lo > 0 or hi < 0: return 'differs'
    if lo > -MARGIN and hi < MARGIN: return f'equivalent (within {MARGIN})'
    return 'inconclusive'
print('border conditions and paired statistics ready')


border conditions and paired statistics ready


In [8]:
# ---------------------------------------------------------------- Flying ArUco v2
FLY = f'{WORK}/flyingarucov2'
if not os.path.isdir(FLY):
    os.makedirs(FLY, exist_ok=True)
    tar = f'{CACHE}/flyingarucov2.tar.gz'
    assert os.path.exists(tar), f'{tar} missing, see N0'
    subprocess.run(['tar','-xzf',tar,'-C',FLY], check=True)
fly_json = sorted(glob.glob(f'{FLY}/**/*.json', recursive=True))
assert len(fly_json) == N0M['flying_aruco_v2']['images'], \
    f'{len(fly_json)} annotations on disk, N0 manifest says {N0M["flying_aruco_v2"]["images"]}'

def bits_for(mid, rot, conv):
    b = CB[mid]; k = rot if conv in (0,1) else -rot
    return np.rot90(b, k) if conv in (0,2) else np.rot90(b, -k)

ref = NumpyDecoder(Wk); probe = []
for jp in fly_json[:60]:
    img = cv2.imread(jp.replace('.json','.jpg'), cv2.IMREAD_GRAYSCALE)
    if img is None: continue
    for m in json.load(open(jp))['markers'][:4]:
        x32, _ = prepare(img, np.zeros(img.shape[:2], np.uint8), get_corners(m))
        if x32 is None: continue
        probe.append((ref(x32), int(m['id']), int(m.get('rot', m.get('rotation', 0)))))
assert len(probe) > 20, f'only {len(probe)} probe markers, Flying ArUco did not load'
scores = {c: float(np.mean([np.mean((p>0.5).astype(np.uint8)==bits_for(i,r,c)) for p,i,r in probe]))
          for c in range(4)}
CONV = max(scores, key=scores.get)
assert scores[CONV] > 0.90, f'no rotation convention agrees ({scores}), labels would be wrong'
print(f'{len(fly_json)} Flying ArUco v2 annotations | rotation convention {CONV} ({scores[CONV]:.4f})')

2500 Flying ArUco v2 annotations | rotation convention 0 (0.9912)


In [9]:
# ---------------------------------------------------------------- training crops
# Same loop, same random draws and same crop order as N14. Only what gets painted differs.
OCC_TRAIN = [0.0, 0.10, 0.20, 0.25, 0.35, 0.45, 0.55]
N_SAMPLES = 5
N_CROPS   = 64840          # the size of every N14 training set; pairing depends on it

def build_training_set(value):
    """'none'          no paint at all                       (arm N)
       'border_random' border ring at a random brightness     (arm BB)
       'data_random'   edge strip clipped to the data region  (arm BD)
    A strip level of 0 means that sample stays clean, exactly as in N14."""
    X, Ybit, Yocc = [], [], []
    t0 = time.time()
    for n, jp in enumerate(fly_json):
        img = cv2.imread(jp.replace('.json','.jpg'), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        key = os.path.basename(jp)
        for m in json.load(open(jp))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            if mid >= 250: continue
            bits = bits_for(mid, int(m.get('rot', m.get('rotation', 0))), CONV)
            rng = np.random.default_rng(zlib.crc32(f'train|{key}|{mid}'.encode()))
            border = data = None
            for L in rng.choice(OCC_TRAIN, N_SAMPLES, replace=False):
                # the same draw as arm E's 'random' set, so BB and BD get E's brightnesses
                v = int(rng.integers(0,256)) if value in ('border_random','data_random') else None
                if value == 'none' or L <= 0:
                    oi, om = img, np.zeros(img.shape[:2], np.uint8)
                elif value == 'border_random':
                    if border is None: border, data = region_masks(cn, img.shape[:2])
                    oi = img.copy(); oi[border > 0] = v; om = border
                elif value == 'data_random':
                    if border is None: border, data = region_masks(cn, img.shape[:2])
                    box = occluder_box(img, cn, float(L), 'edge', key, mid)
                    om = np.zeros(img.shape[:2], np.uint8)
                    if box is not None:
                        a,b,c,d = box; om[b:d, a:c] = 255; om = cv2.bitwise_and(om, data)
                    oi = img.copy(); oi[om > 0] = v
                else:
                    raise ValueError(value)
                x32, occ = prepare(oi, om, cn)
                if x32 is None: continue
                X.append((x32*255).astype(np.uint8)); Ybit.append(bits); Yocc.append(occ)
        if n % 800 == 0: print(f'  {value}: {n}/{len(fly_json)} ({time.time()-t0:.0f}s) crops={len(X)}')
    X = np.stack(X); Ybit = np.stack(Ybit).astype(np.float32); Yocc = np.stack(Yocc).astype(np.float32)
    print(f'  {value}: {len(X)} crops, mean hidden data cells {Yocc.sum((1,2)).mean():.2f}, '
          f'{time.time()-t0:.0f}s')
    return X, Ybit, Yocc

SETS = {v: build_training_set(v) for v in ('none', 'border_random', 'data_random')}

for v, (X, Yb, Yo) in SETS.items():
    assert len(X) == N_CROPS, f'{v}: {len(X)} crops, expected {N_CROPS}. Pairing with N14 would break.'
ref_bits = SETS['none'][1]
for v, (_, Yb, _) in SETS.items():
    assert np.array_equal(Yb, ref_bits), f'{v}: labels differ from the clean set, crop order changed'
assert SETS['none'][2].sum() == 0, 'arm N training crops contain hidden cells'
assert SETS['border_random'][2].sum((1,2)).mean() < 0.5, 'border training hides data cells, ring is wrong'
assert SETS['data_random'][2].sum((1,2)).mean() > 3, 'data region training hides too few cells'
print(f'\nall three sets: {N_CROPS:,} crops, identical labels and order, paint where intended')

  none: 0/2500 (0s) crops=40
  none: 800/2500 (14s) crops=20830
  none: 1600/2500 (29s) crops=41645
  none: 2400/2500 (44s) crops=62275
  none: 64840 crops, mean hidden data cells 0.00, 46s
  border_random: 0/2500 (0s) crops=40
  border_random: 800/2500 (18s) crops=20830
  border_random: 1600/2500 (36s) crops=41645
  border_random: 2400/2500 (54s) crops=62275
  border_random: 64840 crops, mean hidden data cells 0.06, 56s
  data_random: 0/2500 (0s) crops=40
  data_random: 800/2500 (19s) crops=20830
  data_random: 1600/2500 (40s) crops=41645
  data_random: 2400/2500 (60s) crops=62275
  data_random: 64840 crops, mean hidden data cells 6.91, 62s

all three sets: 64,840 crops, identical labels and order, paint where intended


In [10]:
# =================================================================================
# Same training code as N14 and N15 (CODE_TAG n14-v1). One change: an existing checkpoint
# is checked before the training set is touched, so B and E load without their sets.
# =================================================================================
CKPT     = f'{BASE}/ckpt_n14'; os.makedirs(CKPT, exist_ok=True)   # same folder as N14 and N15
CODE_TAG = 'n14-v1'                     # same training code as N14 and N15
SEEDS    = [0, 1, 2, 3, 4]
MARGIN   = 1.0
EPOCHS, PATIENCE, BS = 200, 20, 256

def signature(arm):
    return json.dumps(dict(ARMS[arm], code=CODE_TAG, epochs=EPOCHS, patience=PATIENCE, bs=BS),
                      sort_keys=True, default=str)

def load_arm(arm, seed):
    path = f'{CKPT}/arm_{arm}_s{seed}.pt'
    if not os.path.exists(path): return None
    ck = torch.load(path, map_location=DEV_)
    assert isinstance(ck, dict) and ck.get('sig') == signature(arm), \
        f'{path} was made with other settings or code. It cannot be compared with this notebook.'
    net = Decoder(ARMS[arm]['out_ch'], ARMS[arm]['pool']).to(DEV_)
    net.load_state_dict(ck['state']); net.eval()
    return net
ALLOW_CPU = False
ALLOW_NONDETERMINISM = False

ARMS = {
 # existing, loaded from N14. These two lines must stay exactly as in N14.
 'B':  dict(out_ch=1, mask_bits=False, pool='max', value=127),
 'E':  dict(out_ch=1, mask_bits=False, pool='max', value='random'),
 # new
 'BB': dict(out_ch=1, mask_bits=False, pool='max', value='border_random'),
 'BD': dict(out_ch=1, mask_bits=False, pool='max', value='data_random'),
 'N':  dict(out_ch=1, mask_bits=False, pool='max', value='none'),
}
assert DEV_.type == 'cuda' or ALLOW_CPU, (
    'This is a CPU runtime. Switch to a GPU (Runtime, Change runtime type), '
    'then Restart session and Run all.')

GPU_SETS = {v: (torch.from_numpy(X).to(DEV_), torch.from_numpy(Yb).to(DEV_), torch.from_numpy(Yo).to(DEV_))
            for v, (X, Yb, Yo) in SETS.items()}
idx = np.random.default_rng(42).permutation(N_CROPS); cut = int(0.9*N_CROPS)
TR, VA = torch.from_numpy(idx[:cut]).to(DEV_), torch.from_numpy(idx[cut:]).to(DEV_)

def train_arm(arm, seed, epochs=EPOCHS, patience=PATIENCE, save=True):
    cfg = ARMS[arm]
    if save:
        net = load_arm(arm, seed)
        if net is not None: return net, None, 0.0
    assert cfg['value'] in GPU_SETS, (
        f'{CKPT}/arm_{arm}_s{seed}.pt is missing and its training set is not built here. '
        f'Run N14 first, or copy its checkpoints into {CKPT}.')
    Xg, Bg, Og = GPU_SETS[cfg['value']]
    tr, va = TR, VA
    path = f'{CKPT}/arm_{arm}_s{seed}.pt'
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    rng = np.random.default_rng(seed)
    gen = torch.Generator(device=DEV_); gen.manual_seed(seed)
    net = Decoder(cfg['out_ch'], cfg['pool']).to(DEV_)
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=8)

    def run(ids, train):
        net.train(train)
        order = ids[torch.randperm(len(ids), device=DEV_, generator=gen)] if train else ids
        tot = torch.zeros((), device=DEV_); n = 0
        with torch.set_grad_enabled(train):
            for s in range(0, len(order), BS):
                b  = order[s:s+BS]
                xb = Xg[b].unsqueeze(1).float()/255.0
                bb = Bg[b]; ob = Og[b]
                if train:
                    k, fl = int(rng.integers(0,4)), int(rng.integers(0,3))
                    if k:
                        xb = torch.rot90(xb,k,dims=(-2,-1))
                        bb = torch.rot90(bb,k,dims=(-2,-1)); ob = torch.rot90(ob,k,dims=(-2,-1))
                    if fl == 1:
                        xb, bb, ob = torch.flip(xb,(-1,)), torch.flip(bb,(-1,)), torch.flip(ob,(-1,))
                    elif fl == 2:
                        xb, bb, ob = torch.flip(xb,(-2,)), torch.flip(bb,(-2,)), torch.flip(ob,(-2,))
                logit, elogit = net(xb)
                if cfg['mask_bits']:
                    vis  = 1.0 - ob
                    loss = (F.binary_cross_entropy_with_logits(logit, bb, reduction='none')*vis
                            ).sum()/vis.sum().clamp(min=1)
                else:
                    loss = F.binary_cross_entropy_with_logits(logit, bb)
                if cfg['out_ch'] == 2:
                    loss = loss + F.binary_cross_entropy_with_logits(elogit, ob)
                if train:
                    opt.zero_grad(); loss.backward(); opt.step()
                tot = tot + loss.detach()*len(b); n += len(b)
        return (tot/n).item()

    best, bad, t0, state, ep = 1e9, 0, time.time(), None, 0
    for ep in range(epochs):
        run(tr, True); vl = run(va, False); sch.step(vl)
        if vl < best - 1e-5:
            best, bad = vl, 0
            state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval()
    if save:
        tmp = path + '.tmp'
        torch.save(dict(state=state, sig=signature(arm), best=best, epochs=ep+1), tmp)
        os.replace(tmp, path)
    return net, best, time.time()-t0

for a in ('B', 'E'):
    missing = [s for s in SEEDS if not os.path.exists(f'{CKPT}/arm_{a}_s{s}.pt')]
    assert not missing, f'arm {a} seeds {missing} missing from {CKPT}. Run N14 first.'
print('arm B and arm E checkpoints from N14 are present')

netA, _, secs = train_arm('N', SEEDS[0], epochs=3, patience=99, save=False)
netB, _, _    = train_arm('N', SEEDS[0], epochs=3, patience=99, save=False)
diff = max(float((a-b).abs().max()) for a, b in zip(netA.state_dict().values(), netB.state_dict().values()))
print(f'determinism probe: same seed twice, largest weight difference {diff:.2e}')
assert diff == 0.0 or ALLOW_NONDETERMINISM, (
    'Training is not deterministic. Runtime, Restart session, then Run all.')
print(f'probe: {secs/3:.2f}s per epoch, about {secs/3*150*15/3600:.1f} hours for the 15 new runs')
assert secs/3 < 5, 'too slow, check the device'

# the same seed must give the same starting weights for every one output arm, or pairing is void
w0 = {}
for a in ('N', 'BB'):
    torch.manual_seed(0); torch.cuda.manual_seed_all(0)
    w0[a] = Decoder(1, 'max').to(DEV_).state_dict()
assert all(torch.equal(w0['N'][k], w0['BB'][k]) for k in w0['N']), 'starting weights differ, pairing invalid'
print('pairing check: one output arms start from identical weights for the same seed')

arm B and arm E checkpoints from N14 are present
determinism probe: same seed twice, largest weight difference 0.00e+00
probe: 1.23s per epoch, about 0.8 hours for the 15 new runs
pairing check: one output arms start from identical weights for the same seed


In [11]:
MODELS, TIMES = {}, {}
order = ['B', 'E', 'BB', 'BD', 'N']
total = len(order)*len(SEEDS); k = 0; t_all = time.time()
for arm in order:
    for s in SEEDS:
        k += 1
        net, vl, secs = train_arm(arm, s)
        MODELS[(arm,s)] = net; TIMES[(arm,s)] = secs
        tag = 'loaded from checkpoint' if vl is None else f'trained, val {vl:.4f} ({secs:.0f}s)'
        print(f'  [{k:>2}/{total}] arm {arm:<2} seed {s}: {tag}', flush=True)
print(f'\nall runs ready in {(time.time()-t_all)/60:.1f} min')

  [ 1/25] arm B  seed 0: loaded from checkpoint
  [ 2/25] arm B  seed 1: loaded from checkpoint
  [ 3/25] arm B  seed 2: loaded from checkpoint
  [ 4/25] arm B  seed 3: loaded from checkpoint
  [ 5/25] arm B  seed 4: loaded from checkpoint
  [ 6/25] arm E  seed 0: loaded from checkpoint
  [ 7/25] arm E  seed 1: loaded from checkpoint
  [ 8/25] arm E  seed 2: loaded from checkpoint
  [ 9/25] arm E  seed 3: loaded from checkpoint
  [10/25] arm E  seed 4: loaded from checkpoint
  [11/25] arm BB seed 0: trained, val 0.0309 (91s)
  [12/25] arm BB seed 1: trained, val 0.0305 (61s)
  [13/25] arm BB seed 2: trained, val 0.0336 (44s)
  [14/25] arm BB seed 3: trained, val 0.0324 (49s)
  [15/25] arm BB seed 4: trained, val 0.0306 (69s)
  [16/25] arm BD seed 0: trained, val 0.1843 (108s)
  [17/25] arm BD seed 1: trained, val 0.1834 (61s)
  [18/25] arm BD seed 2: trained, val 0.1817 (96s)
  [19/25] arm BD seed 3: trained, val 0.1791 (84s)
  [20/25] arm BD seed 4: trained, val 0.1778 (91s)
  [21/25]

## Evaluation set

The main grid is identical to N15: clean, and grey, white and black edge occluders at 3 to 18 hidden cells. The border set uses the same markers in the same order.

In [12]:
TARGETS  = [3, 6, 9, 12, 15, 18]
COLOURS  = ['grey', 'white', 'black']
POSITION = 'edge'

cal = []
for vid in DEVV[:2]:
    for f, j in pairs_for(vid)[::STRIDE*8]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for m in json.load(open(j))['markers']:
            cal.append((img, get_corners(m), key, int(m['id'])))
        if len(cal) > 400: break
    if len(cal) > 400: break
def mean_cells(level):
    tot=n=0
    for img, cn, key, mid in cal:
        oi, om = apply_occluder(img, occluder_box(img, cn, level, POSITION, key, mid), 'grey')
        _, occ = prepare(oi, om, cn)
        if occ is None: continue
        tot += occ.sum(); n += 1
    return tot/max(n,1)
curve = [(float(L), mean_cells(float(L))) for L in np.arange(0.02, 0.70, 0.02)]
LEVEL_FOR = {T: min(curve, key=lambda lc: abs(lc[1]-T))[0] for T in TARGETS}
print('severity calibration:', {T: round(LEVEL_FOR[T],2) for T in TARGETS})

CLEAN_KEY = ('clean', 0)
CONDS = [(c,T) for c in COLOURS for T in TARGETS]
EV = {k: {'x':[], 'id':[], 'frame':[], 'occ':[], 'mask':[]} for k in [CLEAN_KEY] + CONDS}
def _add(k, x32, occ, mid, key):
    d = EV[k]
    d['x'].append((x32*255).astype(np.uint8)); d['id'].append(mid)
    d['frame'].append(key); d['occ'].append(int(occ.sum())); d['mask'].append(occ)

t0 = time.time()
for vid in DEVV:
    for f, j in pairs_for(vid)[::STRIDE]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        zero = np.zeros(img.shape[:2], np.uint8)
        for m in json.load(open(j))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            xc, oc = prepare(img, zero, cn)
            if xc is None: continue                      # same corners, so every condition would fail too
            _add(CLEAN_KEY, xc, oc, mid, key)
            for T in TARGETS:
                box = occluder_box(img, cn, LEVEL_FOR[T], POSITION, key, mid)
                for col in COLOURS:
                    oi, om = apply_occluder(img, box, col)
                    x32, occ = prepare(oi, om, cn)
                    _add((col,T), x32, occ, mid, key)
    print(f'  {vid} ({time.time()-t0:.0f}s)')
for k in EV: EV[k] = {kk: np.array(vv) for kk, vv in EV[k].items()}
n = len(EV[CLEAN_KEY]['id'])
assert all(len(EV[k]['id']) == n for k in EV), 'conditions are misaligned'
print(f'\neval set: {n} markers per condition, plus a clean condition')
assert n > 1500, f'only {n} markers, data did not load correctly'


severity calibration: {3: 0.24, 6: 0.3, 9: 0.34, 12: 0.4, 15: 0.46, 18: 0.5}
  video_1 (45s)
  video_2 (79s)
  video_3 (122s)
  video_4 (162s)
  video_5 (195s)

eval set: 12276 markers per condition, plus a clean condition


In [13]:
# Border conditions on the frozen evaluation set, plus the N11 clean scale oracle
BCONDS = ['border_only', 'data_only', 'both']
BEV = {(c,v): {'x':[], 'xo':[], 'id':[], 'frame':[], 'mask':[]} for c in BCONDS for v in INTENS}
t0 = time.time()
for vid in DEVV:
    for f, j in pairs_for(vid)[::STRIDE]:
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        zero = np.zeros(img.shape[:2], np.uint8)
        for m in json.load(open(j))['markers']:
            cn, mid = get_corners(m), int(m['id'])
            xc, _ = prepare(img, zero, cn)
            if xc is None: continue                       # the same rule as the main grid
            sc = crop_scale(img, cn)
            for cond in BCONDS:
                for v in INTENS:
                    oi, om = make_condition(img, cn, cond, v, key, mid)
                    assert oi is not None, 'a condition failed where the clean crop did not'
                    x32, occ = prepare(oi, om, cn)
                    xo, _    = prepare(oi, om, cn, scale=sc)
                    d = BEV[(cond,v)]
                    d['x'].append((x32*255).astype(np.uint8)); d['xo'].append((xo*255).astype(np.uint8))
                    d['id'].append(mid); d['frame'].append(key); d['mask'].append(occ)
    print(f'  {vid} ({time.time()-t0:.0f}s)')
for k in BEV: BEV[k] = {kk: np.array(vv) for kk, vv in BEV[k].items()}
nb_ = len(BEV[('border_only',0)]['id'])
assert nb_ == len(EV[CLEAN_KEY]['id']), 'border set and main set have different markers'
assert all(np.array_equal(BEV[k]['id'], EV[CLEAN_KEY]['id']) for k in BEV), 'marker order differs'
hid = BEV[('border_only',255)]['mask'].sum((1,2)).mean()
assert hid < 0.5, f'border_only hides {hid:.2f} data cells on average, should be about 0'
print(f'\nborder set: {nb_} markers, same markers and order as the main set; '
      f'border_only hides {hid:.3f} data cells on average')

  video_1 (80s)
  video_2 (139s)
  video_3 (216s)
  video_4 (290s)
  video_5 (348s)

border set: 12276 markers, same markers and order as the main set; border_only hides 0.000 data cells on average


In [14]:
# Scoring. Soft decoding for every arm, fair hard decoding on the main grid as well.
def soft_ok(P, ids): return (dec_soft(P) == ids).astype(np.float32)

PRED = {}                        # (arm, seed) -> function from uint8 crops to probabilities
PRED[('P','pub')] = decode_published
for (arm, s), net in MODELS.items():
    PRED[(arm, s)] = (lambda net: (lambda x: decode_torch(net, x)[0]))(net)

R = {}; t0 = time.time()
for (arm, s), f in PRED.items():
    for k in [CLEAN_KEY] + CONDS:
        d = EV[k]; P = f(d['x'])
        R[(arm, s) + k + ('soft',)] = soft_ok(P, d['id'])
        R[(arm, s) + k + ('hard',)] = hard_expected(P, d['id'])
    for (c, v), d in BEV.items():
        R[(arm, s, c, v, 'soft')]        = soft_ok(f(d['x']),  d['id'])
        R[(arm, s, c, v, 'soft_oracle')] = soft_ok(f(d['xo']), d['id'])
print(f'scored {len(R)} combinations in {time.time()-t0:.0f}s')

ARM_SEEDS = {'P': ['pub'], **{a: SEEDS for a in order}}
def acc(arm, s, c, T, dec='soft'): return 100*float(R[(arm, s, c, T, dec)].mean())
def per_seed(arm, c, T, dec='soft'): return np.array([acc(arm, s, c, T, dec) for s in ARM_SEEDS[arm]])
def spread(arm, s, dec='soft'):
    a = [acc(arm, s, 'border_only', v, dec) for v in INTENS]; return max(a) - min(a)
def spreads(arm, dec='soft'): return np.array([spread(arm, s, dec) for s in ARM_SEEDS[arm]])

print('\nREPRODUCTION CHECK, published decoder, border only, soft decoding')
N8_REF  = {0: 92.36, 64: 78.84, 127: 66.82, 192: 56.68, 255: 48.63}
N11_REF = {0: 92.25, 64: 78.74, 127: 66.94, 192: 56.69, 255: 48.70}
for v in INTENS:
    a = acc('P','pub','border_only',v)
    print(f'  brightness {v:>3}: here {a:6.2f}   N8 {N8_REF[v]:6.2f} ({a-N8_REF[v]:+.2f})   '
          f'N11 {N11_REF[v]:6.2f} ({a-N11_REF[v]:+.2f})')
worst = max(min(abs(acc('P','pub','border_only',v)-N8_REF[v]), abs(acc('P','pub','border_only',v)-N11_REF[v]))
            for v in INTENS)
print(f'  closest reference within {worst:.2f} points at every brightness')
if worst > 0.5:
    print('  WARNING: more than 0.5 points from both earlier runs. Do not compare these numbers with N8 or N11.')
print(f'  clean, published decoder: {acc("P","pub","clean",0):.2f} (N8, N11 and N15 all report 99.54)')

scored 1768 combinations in 458s

REPRODUCTION CHECK, published decoder, border only, soft decoding
  brightness   0: here  92.32   N8  92.36 (-0.04)   N11  92.25 (+0.07)
  brightness  64: here  78.68   N8  78.84 (-0.16)   N11  78.74 (-0.06)
  brightness 127: here  66.74   N8  66.82 (-0.08)   N11  66.94 (-0.20)
  brightness 192: here  56.66   N8  56.68 (-0.02)   N11  56.69 (-0.03)
  brightness 255: here  48.71   N8  48.63 (+0.08)   N11  48.70 (+0.01)
  closest reference within 0.08 points at every brightness
  clean, published decoder: 99.54 (N8, N11 and N15 all report 99.54)


## Question 1. Is the border effect something training can remove?

`spread` is the best minus the worst border only accuracy across the five paint brightnesses, computed per seed. A small spread means the decoder no longer cares what colour the border is.

In [15]:
print('QUESTION 1. BORDER ONLY: soft accuracy by paint brightness, mean +- sd over seeds\n')
print(f"{'arm':>5}{'clean':>13}" + ''.join(f'{v:>13}' for v in INTENS) + f"{'spread':>13}")
for arm in ['P'] + order:
    cl = per_seed(arm, 'clean', 0)
    line = f'{arm:>5}{cl.mean():>8.2f}+-{cl.std(ddof=1) if len(cl)>1 else 0:<3.1f}'
    for v in INTENS:
        a = per_seed(arm, 'border_only', v)
        line += f'{a.mean():>8.2f}+-{(a.std(ddof=1) if len(a)>1 else 0):<3.1f}'
    sp = spreads(arm)
    line += f'{sp.mean():>8.2f}+-{(sp.std(ddof=1) if len(sp)>1 else 0):<3.1f}'
    print(line)

print('\nPAIRED DIFFERENCES IN SPREAD (seed s of one arm against seed s of the other)\n')
Q1 = []
for a1, a2, q in [('BB','N','PRIMARY: border training against no paint, same recipe'),
                  ('BD','N','data region training against no paint'),
                  ('BB','BD','border training against data region training'),
                  ('E', 'N','edge strip training against no paint'),
                  ('BB','E','border training against edge strip training')]:
    m, lo, hi = paired_t(spreads(a1), spreads(a2)); v = verdict(lo, hi)
    Q1.append(dict(pair=f'{a1}-{a2}', question=q, mean=m, lo=lo, hi=hi, verdict=v))
    print(f'  {a1:>2} minus {a2:<2} {m:>8.2f}   [{lo:>7.2f}, {hi:>7.2f}]   {v:<26} {q}')

print('\nIS OUR CLEAN RECIPE AS BORDER SENSITIVE AS THE PUBLISHED DECODER? published minus N, spread')
pm, plo, phi = one_sample_t(spreads('P')[0] - spreads('N'))
print(f'  published {spreads("P")[0]:.2f}, N {spreads("N").mean():.2f}+-{spreads("N").std(ddof=1):.2f}, '
      f'difference {pm:.2f} [{plo:.2f}, {phi:.2f}] {verdict(plo, phi)}')
nsp = spreads('N').mean()
NOTE_N = ('N is border sensitive, so there is an effect to remove.' if nsp >= 10 else
          f'N is only mildly border sensitive (spread {nsp:.1f}). The border effect is mostly a property '
          'of the published decoder, not of clean training in general, so say that in the paper.')
print('  ' + NOTE_N)

bb = spreads('BB').mean(); r = [x for x in Q1 if x['pair'] == 'BB-N'][0]
if r['verdict'] == 'differs' and r['mean'] < 0 and bb < 5:
    V1 = 'Border training REMOVES the border effect. It is a gap in what the decoder was shown.'
elif r['verdict'] == 'differs' and r['mean'] < 0:
    V1 = f'Border training REDUCES the border effect but does not remove it (spread {bb:.1f} points).'
else:
    V1 = 'Border training does NOT detectably change the border effect.'
V1 = V1 + ' ' + NOTE_N
print(f'\nVERDICT Q1 (rule fixed in advance): {V1}')

print('\nCLEAN SCALE ORACLE (N11) on border only: spread with the published scaling and with the')
print('scale taken from the clean image of the same marker\n')
print(f"{'arm':>5}{'normal':>10}{'oracle':>10}{'reduction':>12}")
ORACLE = {}
for arm in ['P'] + order:
    sn, so = spreads(arm).mean(), spreads(arm, 'soft_oracle').mean()
    ORACLE[arm] = dict(normal=sn, oracle=so, reduction_pct=100*(1-so/sn) if sn > 0 else float('nan'))
    print(f'{arm:>5}{sn:>10.2f}{so:>10.2f}{ORACLE[arm]["reduction_pct"]:>11.1f}%')
print('N11 reported 43.5, 24.0 and 44.8 percent for the published decoder.')
print('Write these as reductions in spread after an intervention, never as a share of the effect explained.')

print('\nDOES BORDER TRAINING COST ANYTHING ELSEWHERE? BB minus N, paired (same recipe)\n')
COST = []
for col, T in [('clean', 0)] + [(c, t) for c in COLOURS for t in (6, 12, 18)]:
    m, lo, hi = paired_t(per_seed('BB', col, T), per_seed('N', col, T))
    cells = 0.0 if col == 'clean' else EV[(col,T)]['occ'].mean()
    COST.append(dict(colour=col, target=T, mean=m, lo=lo, hi=hi, verdict=verdict(lo, hi)))
    print(f'  {col:>5} {cells:>5.1f} cells  {m:>7.2f}   [{lo:>7.2f}, {hi:>7.2f}]   {verdict(lo, hi)}')

QUESTION 1. BORDER ONLY: soft accuracy by paint brightness, mean +- sd over seeds

  arm        clean            0           64          127          192          255       spread
    P   99.54+-0.0   92.32+-0.0   78.68+-0.0   66.74+-0.0   56.66+-0.0   48.71+-0.0   43.61+-0.0
    B   99.60+-0.6   89.15+-2.0   93.45+-3.4   86.89+-6.7   81.62+-7.6   77.12+-7.6   16.89+-5.1
    E   99.82+-0.2   87.85+-3.0   92.61+-2.9   83.79+-4.5   77.87+-4.2   73.78+-4.3   19.38+-2.6
   BB   99.68+-0.1   96.34+-1.4   99.17+-0.5   99.20+-0.6   98.26+-1.4   95.99+-2.8    4.50+-1.7
   BD   99.64+-0.1   95.69+-2.9   92.13+-2.4   83.31+-3.4   77.10+-3.3   72.79+-3.9   23.04+-3.7
    N   99.56+-0.3   96.02+-0.9   71.81+-2.3   59.36+-3.2   48.91+-4.3   40.45+-5.2   55.58+-5.8

PAIRED DIFFERENCES IN SPREAD (seed s of one arm against seed s of the other)

  BB minus N    -51.08   [ -56.53,  -45.63]   differs                    PRIMARY: border training against no paint, same recipe
  BD minus N    -32.54   [ -42.

## Question 2. Is finding 4 caused by occlusion training or by the recipe?

In [16]:
print('QUESTION 2. CONTROL ARM. Soft accuracy, mean +- sd over seeds (P is one fixed model)\n')
show = ['P', 'N', 'B', 'E']
for col in ['clean'] + COLOURS:
    Ts = [0] if col == 'clean' else TARGETS
    print(f'--- {col}')
    print(f"{'cells':>7}" + ''.join(f'{a:>14}' for a in show))
    for T in Ts:
        cells = 0.0 if col == 'clean' else EV[(col,T)]['occ'].mean()
        line = f'{cells:>7.1f}'
        for a in show:
            v = per_seed(a, col, T); line += f'{v.mean():>9.2f}+-{(v.std(ddof=1) if len(v)>1 else 0):<3.1f}'
        print(line)
    print()

print('PAIRED N minus B: occlusion training with the recipe held fixed (positive means N is better)\n')
Q2 = []
for col in COLOURS:
    for T in TARGETS:
        m, lo, hi = paired_t(per_seed('N', col, T), per_seed('B', col, T))
        Q2.append(dict(colour=col, target=T, cells=float(EV[(col,T)]['occ'].mean()),
                       mean=m, lo=lo, hi=hi, verdict=verdict(lo, hi)))
        print(f'  {col:>5} {EV[(col,T)]["occ"].mean():>5.1f} cells  {m:>7.2f}   [{lo:>7.2f}, {hi:>7.2f}]   {verdict(lo, hi)}')

print('\nPUBLISHED DECODER minus N: the recipe alone (interval over N seeds)\n')
RECIPE = []
for col in ['clean'] + COLOURS:
    for T in ([0] if col == 'clean' else TARGETS):
        d = per_seed('P', col, T)[0] - per_seed('N', col, T)
        m, lo, hi = one_sample_t(d)
        RECIPE.append(dict(colour=col, target=T, mean=m, lo=lo, hi=hi, verdict=verdict(lo, hi)))
        print(f'  {col:>5} {T:>2}  {m:>7.2f}   [{lo:>7.2f}, {hi:>7.2f}]   {verdict(lo, hi)}')

blk = [r for r in Q2 if r['colour'] == 'black']
n_better = sum(1 for r in blk if r['verdict'] == 'differs' and r['mean'] > 0)
n_worse  = sum(1 for r in blk if r['verdict'] == 'differs' and r['mean'] < 0)
if n_better >= 4 and n_worse == 0:
    V2 = (f'N beats B on black in {n_better} of 6 severities. Training on grey occluders makes black '
          'worse, and finding 4 holds with the recipe controlled.')
elif n_better == 0:
    V2 = ('N does not beat B on black. The published decoder\'s lead over B on black comes from the '
          'training recipe, not from occlusion training. Finding 4 must be rewritten.')
else:
    V2 = (f'Mixed: N beats B on black in {n_better} of 6 severities and loses in {n_worse}. '
          'Report finding 4 per severity, not as a single claim.')
print(f'\nVERDICT Q2 (rule fixed in advance): {V2}')

QUESTION 2. CONTROL ARM. Soft accuracy, mean +- sd over seeds (P is one fixed model)

--- clean
  cells             P             N             B             E
    0.0    99.54+-0.0    99.56+-0.3    99.60+-0.6    99.82+-0.2

--- grey
  cells             P             N             B             E
    3.2    85.70+-0.0    74.78+-3.0    97.21+-2.2    97.47+-1.5
    5.7    76.33+-0.0    70.22+-2.6    94.87+-3.0    95.88+-1.7
    8.5    65.40+-0.0    65.42+-1.2    92.56+-3.5    93.82+-2.2
   11.5    49.22+-0.0    59.88+-0.4    90.23+-3.6    91.51+-2.9
   15.6    31.97+-0.0    51.75+-1.0    85.82+-3.3    87.38+-2.5
   18.0    22.59+-0.0    40.59+-1.8    80.11+-3.7    82.06+-3.4

--- white
  cells             P             N             B             E
    3.2    76.71+-0.0    61.06+-3.4    90.55+-5.6    91.99+-3.7
    5.7    63.48+-0.0    54.84+-3.8    86.18+-5.2    88.68+-3.1
    8.5    51.08+-0.0    48.12+-2.0    83.03+-5.3    84.98+-3.5
   11.5    36.19+-0.0    40.34+-0.7    80.27+-4.8  

In [17]:
res = dict(code=CODE_TAG, seeds=SEEDS, margin=MARGIN, intens=INTENS,
           arms={k: {kk: str(vv) for kk, vv in v.items()} for k, v in ARMS.items()},
           protocol=dict(views=DEVV, stride=STRIDE, markers=int(len(EV[CLEAN_KEY]['id']))),
           seconds={f'{a}|{s}': float(t) for (a, s), t in TIMES.items()},
           verdict_q1=V1, verdict_q2=V2, q1=Q1, oracle=ORACLE, cost=COST, q2=Q2, recipe=RECIPE)
with open(f'{ART}/n16_dev.json', 'w') as fh: json.dump(res, fh, indent=1)
uni = {f'{a}|{s}|{c}|{T}|{dec}': float(v.mean()) for (a, s, c, T, dec), v in R.items()}
with open(f'{ART}/n16_unified_dev.json', 'w') as fh: json.dump(uni, fh, indent=1)
print('saved', f'{ART}/n16_dev.json', 'and', f'{ART}/n16_unified_dev.json')

saved /content/drive/MyDrive/aruco_p2/artifacts/n16_dev.json and /content/drive/MyDrive/aruco_p2/artifacts/n16_unified_dev.json


## Results workbook

Rebuilds `all_results.xlsx` from every saved result file. Two fixes over N15: `manifest.json` now loads (it failed on a duplicate column name), and N0 is counted as saved, so COVERAGE no longer shows a false alarm for it.

In [18]:
# ---------------------------------------------------------------------------------
# Collect every saved result (all .json files in the artifacts folder) into one Excel
# workbook. One tab per file, plus an INDEX tab listing every file and a COVERAGE tab
# showing which notebooks have no saved results yet. Safe to rerun at any time.
# ---------------------------------------------------------------------------------
import re, glob, json, math, datetime
import numpy as np, pandas as pd
from openpyxl.styles import Font

XLSX = f'{ART}/all_results.xlsx'
NOTEBOOKS = list(range(0, 18))          # notebooks expected to have saved something
MAX_ROWS = 1_000_000

# how to name the parts of keys written as 'a|b|c'
KNOWN = {
    'accuracy':        ['arm', 'seed', 'colour', 'target'],
    'accuracy_soft':   ['arm', 'seed', 'colour', 'target'],
    'seconds':         ['arm', 'seed'],
    'cap':             ['seed', 'eps', 'colour', 'target', 'split'],
    'n13_unified_dev': ['arm', 'seed', 'colour', 'target', 'decoder'],
    'n14_unified_dev': ['arm', 'seed', 'colour', 'target', 'decoder'],
    'n15_unified_dev': ['arm', 'seed', 'colour', 'target', 'decoder'],
    'n16_unified_dev': ['arm', 'seed', 'colour', 'target', 'decoder'],
}
NOTES = {   # results that later notebooks replaced
    'n13_seeds_dev':   'superseded by N14 (N13 mixed checkpoints from two code versions)',
    'n13_unified_dev': 'superseded by N14 (N13 mixed checkpoints from two code versions)',
}

def is_scalar(x): return x is None or isinstance(x, (str, int, float, bool))

def num(s):
    try: return pd.to_numeric(s)
    except (ValueError, TypeError): return s

def cols_for(name, stem, n):
    for part in [p for p in reversed(name.split('.'))] + [stem]:
        if part in KNOWN and len(KNOWN[part]) == n: return KNOWN[part]
    return [f'key{i+1}' for i in range(n)]

def split_keys(d, name, stem):
    rows = [str(k).split('|') for k in d]
    n = max(len(r) for r in rows)
    names = cols_for(name, stem, n)
    df = pd.DataFrame([r + [None]*(n-len(r)) for r in rows], columns=names)
    for c in names: df[c] = num(df[c])
    df['value'] = list(d.values())
    return df

def tables(obj, name, stem):
    """Yield (title, DataFrame) pairs for any JSON value."""
    if isinstance(obj, list):
        if obj and all(isinstance(x, dict) for x in obj):
            yield name, pd.json_normalize(obj, sep='.')
        elif all(is_scalar(x) for x in obj):
            yield name, pd.DataFrame({name.split('.')[-1] or 'value': obj})
        else:
            yield name, pd.DataFrame(obj)
        return
    if not isinstance(obj, dict):
        yield name, pd.DataFrame({'value': [obj]}); return

    scal = {k: v for k, v in obj.items() if is_scalar(v)}
    rest = {k: v for k, v in obj.items() if not is_scalar(v)}
    if scal:
        if len(scal) >= 3 and all('|' in str(k) for k in scal):
            yield name, split_keys(scal, name, stem)
        else:
            yield name, pd.DataFrame({'key': [str(k) for k in scal], 'value': list(scal.values())})
    # a dict of small flat dicts (e.g. arm settings) becomes one table, one row each
    flat_dicts = {k: v for k, v in rest.items() if isinstance(v, dict) and v
                  and all(is_scalar(x) for x in v.values()) and not any('|' in str(x) for x in v)}
    if len(flat_dicts) >= 2:
        df = pd.DataFrame.from_dict(flat_dicts, orient='index')
        col = 'name' if 'name' not in df.columns else 'item'   # manifest.json has a 'name' field
        df.insert(0, col, df.index.astype(str)); yield name, df.reset_index(drop=True)
        rest = {k: v for k, v in rest.items() if k not in flat_dicts}
    for k, v in rest.items():
        yield from tables(v, f'{name}.{k}' if name else str(k), stem)

def clean(df):
    df = df.copy()
    for c in df.columns:
        if df[c].map(lambda x: isinstance(x, (list, dict))).any():
            df[c] = df[c].map(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)
        if df[c].dtype.kind == 'f':
            df[c] = df[c].where(np.isfinite(df[c]), None)
    df.columns = [str(c) for c in df.columns]
    return df.head(MAX_ROWS)

def sheet_name(stem, used):
    s = re.sub(r'[\[\]:*?/\\]', '_', stem)[:31] or 'sheet'
    base, i = s, 2
    while s.lower() in used:
        s = f'{base[:28]}_{i}'; i += 1
    used.add(s.lower()); return s

files = sorted(set(glob.glob(f'{ART}/**/*.json', recursive=True)) | set(glob.glob(f'{BASE}/*.json')))
def nb_of(path):
    if os.path.basename(path).lower() in ('manifest.json', 'splits.json'): return 0
    m = re.match(r'^n(\d+)', os.path.basename(path).lower())
    return int(m.group(1)) if m else None
files.sort(key=lambda p: (nb_of(p) if nb_of(p) is not None else -1, os.path.basename(p)))
print(f'found {len(files)} result files under {ART}\n')

index, used, bad = [], {'index', 'coverage'}, []
with pd.ExcelWriter(XLSX, engine='openpyxl') as xw:
    pd.DataFrame({'building': ['...']}).to_excel(xw, sheet_name='INDEX', index=False)
    pd.DataFrame({'building': ['...']}).to_excel(xw, sheet_name='COVERAGE', index=False)
    for path in files:
        stem = os.path.splitext(os.path.basename(path))[0]
        try:
            obj = json.load(open(path))
            parts = [(t, clean(df)) for t, df in tables(obj, '', stem) if len(df.columns)]
        except Exception as e:
            bad.append((path, repr(e))); print(f'  could not read {path}: {e!r}'); continue
        sh = sheet_name(stem, used); row = 0
        for title, df in parts:
            df.to_excel(xw, sheet_name=sh, startrow=row + 1, index=False)
            ws = xw.sheets[sh]
            ws.cell(row=row + 1, column=1, value=title or stem).font = Font(bold=True)
            row += len(df) + 4
        index.append(dict(notebook=f'N{nb_of(path)}' if nb_of(path) is not None else 'setup',
                          sheet=sh, file=os.path.relpath(path, BASE),
                          modified=datetime.datetime.fromtimestamp(os.path.getmtime(path)).strftime('%Y-%m-%d %H:%M'),
                          size_kb=round(os.path.getsize(path)/1024, 1), tables=len(parts),
                          rows=int(sum(len(d) for _, d in parts)), note=NOTES.get(stem, '')))
        print(f'  {sh:<32} {len(parts):>3} tables {index[-1]["rows"]:>8} rows   {index[-1]["file"]}')

    idx = pd.DataFrame(index)
    have = {nb_of(p) for p in files if nb_of(p) is not None}
    cov = pd.DataFrame([dict(notebook=f'N{n}', files=int(sum(1 for p in files if nb_of(p) == n)),
                             status='saved' if n in have else
                                    'NOTHING SAVED: rerun this notebook, or its results exist only as printed output')
                        for n in NOTEBOOKS])
    idx.to_excel(xw, sheet_name='INDEX', index=False)
    cov.to_excel(xw, sheet_name='COVERAGE', index=False)
    for name in ('INDEX', 'COVERAGE'):
        ws = xw.sheets[name]
        for cell in ws[1]: cell.font = Font(bold=True)
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(80, max(10, max(len(str(c.value or '')) for c in col) + 2))

print(f'\nsaved {XLSX}')
print('\nCOVERAGE')
print(cov.to_string(index=False))
if bad: print('\nfiles that could not be read:', *bad, sep='\n  ')
try:
    from google.colab import files as _files
    _files.download(XLSX)
except Exception:
    pass


found 25 result files under /content/drive/MyDrive/aruco_p2/artifacts

  manifest                          14 tables       56 rows   artifacts/manifest.json
  splits                             3 tables        8 rows   artifacts/splits.json
  n1a_headroom                       3 tables       23 rows   artifacts/n1a_headroom.json
  n2_occlusion                       6 tables       32 rows   artifacts/n2_occlusion.json
  n3_results                         4 tables       21 rows   artifacts/n3_results.json
  n3b_results                        3 tables       23 rows   artifacts/n3b_results.json
  n4_controlled                    188 tables      506 rows   artifacts/n4_controlled.json
  n5_significance                    4 tables      109 rows   artifacts/n5_significance.json
  n6_mechanism_mitigation            4 tables       20 rows   artifacts/n6_mechanism_mitigation.json
  n7_error_localisation              4 tables       23 rows   artifacts/n7_error_localisation.json
  n8_border_causal

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>